# Big Data with DuckDB in R

## What you'll learn
- What DuckDB is and why it's useful
- How to run SQL queries on CSV files directly from R
- SQL basics: SELECT, WHERE, GROUP BY, ORDER BY, and aggregate functions
- Using dplyr with DuckDB via the `dbplyr` package
- When DuckDB is the right choice

## Prerequisites
- Completed Notebooks 01 and 02
- Generated the large dataset: `python scripts/generate_large_data.py`

## What Is DuckDB?

DuckDB is an **embedded analytical database**. Let's unpack that:

- **Database:** It stores and queries data using SQL (Structured Query Language)
- **Analytical:** It's optimized for the kind of queries data analysts run — aggregations, filters, joins across large tables
- **Embedded:** It runs inside your R session — no separate server to install or manage

Think of DuckDB as a supercharged calculator that speaks SQL. You point it at a CSV file, write a query, and it returns results very quickly — often faster than loading the file into R first.

DuckDB can also read Parquet files and work with data that doesn't fit in memory.

## Setup

The `duckdb` R package is not available through conda-forge, so it is not included in the `codingcs` environment. Run the cell below to install it from CRAN (R's standard package repository). This only needs to be done once.

After the first install, **comment out** the line by placing a `#` at the beginning — this turns it into a **comment**, which R ignores when running the cell. Comments are useful for leaving notes in your code or disabling lines you don't want to run anymore. You have already seen `#` used in earlier notebooks for short annotations next to code.

In [ ]:
install.packages("duckdb", repos = c("https://duckdb.r-universe.dev", "https://cloud.r-project.org"))
# After installing, comment out the line above so it looks like this:
# install.packages("duckdb", repos = c("https://duckdb.r-universe.dev", "https://cloud.r-project.org"))

In [ ]:
library(duckdb)
library(DBI)
library(dplyr)

## Connecting to DuckDB

To use DuckDB, you create a **connection**. This is like opening a channel between R and the DuckDB engine.

The `DBI` package provides a standard interface for talking to databases in R. `dbConnect()` opens the connection; `dbDisconnect()` closes it.

In [ ]:
# Create an in-memory DuckDB connection
con <- dbConnect(duckdb())
con

## Querying CSV Files Directly

One of DuckDB's best features: you can run SQL queries directly on CSV files without loading them into R first. The function `read_csv_auto()` (inside SQL) handles this.

In [ ]:
# Read the first 5 rows of our sales data
dbGetQuery(con, "
  SELECT *
  FROM read_csv_auto('../data/sales.csv')
  LIMIT 5
")

## SQL Basics

SQL (Structured Query Language) is the standard language for working with databases. If you learn it, you can use it with DuckDB, PostgreSQL, MySQL, BigQuery, and many other systems.

Here are the essential SQL commands:

| SQL Keyword | What it does | dplyr equivalent |
|-------------|-------------|------------------|
| `SELECT` | Choose columns | `select()` |
| `WHERE` | Filter rows | `filter()` |
| `GROUP BY` | Group rows for aggregation | `group_by()` |
| `ORDER BY` | Sort results | `arrange()` |
| `COUNT()` | Count rows | `n()` |
| `SUM()` | Add up values | `sum()` |
| `AVG()` | Calculate average | `mean()` |
| `LIMIT` | Restrict number of results | `head()` |

Let's try each one.

### SELECT — Choose Columns

In [ ]:
# Select specific columns
dbGetQuery(con, "
  SELECT date, product, category, unit_price
  FROM read_csv_auto('../data/sales.csv')
  LIMIT 10
")

### WHERE — Filter Rows

In [ ]:
# Only electronics
dbGetQuery(con, "
  SELECT product, unit_price, quantity
  FROM read_csv_auto('../data/sales.csv')
  WHERE category = 'Electronics'
  LIMIT 10
")

In [ ]:
# Multiple conditions
dbGetQuery(con, "
  SELECT product, unit_price, region
  FROM read_csv_auto('../data/sales.csv')
  WHERE category = 'Electronics' AND unit_price > 500
  LIMIT 10
")

### GROUP BY + Aggregation Functions

In [ ]:
# Count transactions per category
dbGetQuery(con, "
  SELECT category, COUNT(*) AS num_transactions
  FROM read_csv_auto('../data/sales.csv')
  GROUP BY category
  ORDER BY num_transactions DESC
")

In [ ]:
# Total revenue per category
dbGetQuery(con, "
  SELECT
    category,
    SUM(quantity * unit_price) AS total_revenue,
    AVG(unit_price) AS avg_price,
    COUNT(*) AS num_transactions
  FROM read_csv_auto('../data/sales.csv')
  GROUP BY category
  ORDER BY total_revenue DESC
")

### ORDER BY — Sort Results

In [ ]:
# Top 10 most expensive individual transactions
dbGetQuery(con, "
  SELECT product, category, quantity, unit_price,
         quantity * unit_price AS total_price
  FROM read_csv_auto('../data/sales.csv')
  ORDER BY total_price DESC
  LIMIT 10
")

## Using DuckDB on the Large Dataset

Now let's use DuckDB on the 100K-row dataset. DuckDB can query it without loading it into R's memory.

In [ ]:
# Revenue by region and category on the large dataset
dbGetQuery(con, "
  SELECT
    region,
    category,
    ROUND(SUM(quantity * unit_price), 2) AS total_revenue,
    COUNT(*) AS transactions
  FROM read_csv_auto('../data/sales_large.csv')
  GROUP BY region, category
  ORDER BY region, total_revenue DESC
")

In [ ]:
# Time a complex query on the large dataset
system.time({
  result <- dbGetQuery(con, "
    SELECT
      category,
      product,
      COUNT(*) AS times_sold,
      ROUND(AVG(unit_price), 2) AS avg_price,
      SUM(quantity) AS total_qty
    FROM read_csv_auto('../data/sales_large.csv')
    GROUP BY category, product
    ORDER BY total_qty DESC
  ")
})
result

## Using dplyr with DuckDB (dbplyr)

If you prefer dplyr syntax over SQL, the `dbplyr` package translates your dplyr code into SQL and runs it on DuckDB. You get the speed of DuckDB with the syntax you already know.

First, you register the CSV file as a virtual table, then use `tbl()` to create a dplyr-compatible reference.

In [ ]:
library(dbplyr)

# Register the CSV as a DuckDB view
dbExecute(con, "CREATE OR REPLACE VIEW sales AS SELECT * FROM read_csv_auto('../data/sales_large.csv')")

# Create a dplyr reference to the DuckDB table
sales_db <- tbl(con, "sales")
sales_db

In [ ]:
# Now use dplyr verbs — they get translated to SQL and run on DuckDB
sales_db |>
  filter(category == "Electronics") |>
  group_by(product) |>
  summarize(
    avg_price = mean(unit_price, na.rm = TRUE),
    total_sold = sum(quantity, na.rm = TRUE)
  ) |>
  arrange(desc(total_sold)) |>
  collect()

In [ ]:
# See the SQL that dbplyr generates
sales_db |>
  filter(category == "Electronics") |>
  group_by(product) |>
  summarize(avg_price = mean(unit_price, na.rm = TRUE)) |>
  show_query()

## Creating Tables from R Data Frames

You can also push an R data frame into DuckDB and query it with SQL.

In [ ]:
# Load employees into DuckDB
employees <- read.csv("../data/employees.csv")
dbWriteTable(con, "employees", employees, overwrite = TRUE)

# Query it with SQL
dbGetQuery(con, "
  SELECT department,
         ROUND(AVG(salary), 2) AS avg_salary,
         COUNT(*) AS count
  FROM employees
  GROUP BY department
  ORDER BY avg_salary DESC
")

## When to Use DuckDB

DuckDB is the right choice when:
- You know SQL (or want to learn it — it's a very valuable skill)
- You need fast analytical queries on local files
- You want to query CSV/Parquet files without loading them into R
- You're doing aggregations and summaries on medium-to-large data

DuckDB is **not** the right choice when:
- Your data is distributed across a cluster (use Spark)
- You need to do complex R-specific transformations (use Arrow + dplyr)
- Your data is small and you're already comfortable with dplyr

## Disconnecting

Always close your database connection when done.

In [ ]:
dbDisconnect(con, shutdown = TRUE)

---
## Summary

- **DuckDB** is a fast, embedded SQL database that runs inside R
- It can query CSV and Parquet files **directly** — no need to load them into R first
- Use `dbGetQuery()` for SQL, or `tbl()` + dplyr via dbplyr
- SQL is a universal language for data — learning it is a great investment
- DuckDB is lightweight (no Java needed) and fast for analytical workloads

**Next up:** [06 - Data Visualization](06-visualization.ipynb) — data visualization with ggplot2.